In [1]:
import pandas as pd
import numpy as np

# --- Загрузка данных ---
# Файл с предсказаниями: колонки без названий, поэтому задаём имена сами
df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=0)  # header=0 если есть заголовок? Уточним
# Если в файле нет заголовка, используйте header=None
# df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=None)

# Файл Б с реальными высотами
df_true = pd.read_csv("cup_it_example_src_B_cleaned.csv", usecols=["id", "height_numeric"])

# --- Объединение по id (только те, для которых есть предсказания) ---
merged = pd.merge(df_true, df_pred, on="id", how="inner")

# Убедимся, что высоты числовые
merged["height_numeric"] = pd.to_numeric(merged["height_numeric"], errors="coerce")
merged["pred"] = pd.to_numeric(merged["pred"], errors="coerce")

# Удаляем строки с пропусками, если они есть
merged = merged.dropna(subset=["height_numeric", "pred"])

# Вычисляем разницу (предсказание - реальность)
merged["diff"] = merged["pred"] - merged["height_numeric"]

# --- Создание интервалов по реальной высоте ---
# Задаём границы интервалов. Например, от 0 до 100 с шагом 10.
# При необходимости измените bins под ваши данные (можно вычислить автоматически)
bins = list(range(0, 101, 10))  # [0, 10, 20, ..., 100]
labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

# Добавляем колонку с интервалом
merged["height_interval"] = pd.cut(merged["height_numeric"], bins=bins, labels=labels, right=False)

# --- Группировка по интервалам и расчёт метрик ---
def rmse(x):
    return np.sqrt(np.mean(x ** 2))

metrics = merged.groupby("height_interval").agg(
    count=("id", "count"),
    MAE=("diff", lambda x: np.abs(x).mean()),
    RMSE=("diff", rmse),
    mean_diff=("diff", "mean")  # средняя разница (смещение)
).reset_index()

# Выводим результат
print(metrics)

# Сохраняем в CSV для дальнейшего анализа
metrics.to_csv("оценка_по_интервалам.csv", index=False)

# Дополнительно: можно вывести общую статистику для всех зданий
#print("\nОбщая статистика для всей выборки:")
#print(f"Количество зданий: {len(merged)}")
#print(f"Общая MAE: {np.abs(merged['diff']).mean():.2f}")
#print(f"Общая RMSE: {np.sqrt((merged['diff']**2).mean()):.2f}")

  height_interval  count        MAE       RMSE  mean_diff
0            0-10   7017  13.666397  20.568072  13.575134
1           10-20   5203  10.605048  17.060155   7.431081
2           20-30   1901  12.227991  15.518878  -2.460408
3           30-40   2798  16.866435  18.799462 -12.461319
4           40-50    864  26.769499  28.395931 -25.247398
5           50-60    890  35.118720  36.550207 -33.970442
6           60-70    220  42.484007  44.511534 -41.266716
7           70-80    129  56.327946  57.618260 -56.006395
8           80-90    199  63.761553  64.941900 -63.761553
9          90-100     16  70.959812  71.872343 -70.959812


C:\Users\Анастасия\AppData\Local\Temp\ipykernel_9524\573089437.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  metrics = merged.groupby("height_interval").agg(
